# 五种主流签名协议原理与实现 — 自学课程

**分类**：密码协议设计专项

**内容简介**：详解群签名、环签名等五种签名协议的原理、数学核心，提供Python实现与场景演示



## 学习目标

- 用“消息序列”描述协议
- 写出最小可运行模拟
- 识别重放/中间人/身份绑定等常见风险
- 通过小实验理解修补策略



## 工具箱（标准库）

- Hash：$H(m)=\mathrm{SHA256}(m)$
- MAC：$t=\mathrm{HMAC}_k(m)$
- Nonce：$n\leftarrow\{0,1\}^{\lambda}$



In [ ]:
# Step 0：基础密码工具
import hashlib, hmac, secrets, time
def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()
def hmac_sha256(key: bytes, msg: bytes) -> bytes:
    return hmac.new(key, msg, hashlib.sha256).digest()
def randbytes(n: int) -> bytes:
    return secrets.token_bytes(n)

print(len(randbytes(16)))
# 预期输出：16



## 自检小测

1) 为什么协议里经常需要 nonce？
2) 为什么建议用 hmac.compare_digest 做比较？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



# Step 1：签名三算法

- KeyGen
- Sign
- Verify

$$\mathrm{Verify}_{pk}(m,\sigma)=1$$ 表示签名有效。



> 提醒：这里的 RSA 实现是教学版，不包含安全填充，不可用于真实场景。



In [ ]:
# Step 1：Miller-Rabin + RSA KeyGen（教学）
def is_probable_prime(n: int, k: int = 8) -> bool:
    if n < 2:
        return False
    small=[2,3,5,7,11,13,17,19,23,29]
    for p in small:
        if n%p==0:
            return n==p
    d=n-1; s=0
    while d%2==0:
        d//=2; s+=1
    for _ in range(k):
        a=secrets.randbelow(n-3)+2
        x=pow(a,d,n)
        if x in (1,n-1):
            continue
        for __ in range(s-1):
            x=pow(x,2,n)
            if x==n-1:
                break
        else:
            return False
    return True

def gen_prime(bits: int) -> int:
    while True:
        n=secrets.randbits(bits) | 1 | (1<<(bits-1))
        if is_probable_prime(n):
            return n

def egcd(a: int, b: int):
    if b==0:
        return a,1,0
    g,x1,y1=egcd(b,a%b)
    return g,y1,x1-(a//b)*y1

def inv_mod(a: int, n: int):
    g,x,_=egcd(a,n)
    return None if g!=1 else x%n

def rsa_keygen(bits=256):
    p=gen_prime(bits//2); q=gen_prime(bits//2)
    n=p*q; phi=(p-1)*(q-1)
    e=65537
    d=inv_mod(e,phi)
    if d is None:
        return rsa_keygen(bits)
    return (n,e),(n,d)

pk,sk=rsa_keygen(256)
print("n_bits=", pk[0].bit_length())
# 预期输出：256（或接近）



In [ ]:
# Step 2：签名与验证（hash 后签名）
def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def rsa_sign(sk, msg: bytes) -> int:
    n,d=sk
    h=int.from_bytes(sha256(msg),'big')
    return pow(h,d,n)

def rsa_verify(pk, msg: bytes, sig: int) -> bool:
    n,e=pk
    h=int.from_bytes(sha256(msg),'big')
    return h == pow(sig,e,n)

msg=b"hello"
sig=rsa_sign(sk,msg)
print(rsa_verify(pk,msg,sig), rsa_verify(pk,b"x",sig))
# 预期输出：True False



# Step 3：协议层面的签名使用

常见模式：

- 对“消息 + 上下文”签名：$\sigma=\mathrm{Sign}(ctx\|m)$
- ctx 包含：用途标签、版本、链 ID、nonce 等

> 这样可避免签名被挪用到别的场景（cross-protocol attack）。



In [ ]:
# Step 3：上下文绑定签名
def sign_with_ctx(sk, ctx: str, m: bytes) -> int:
    return rsa_sign(sk, ctx.encode()+b"|" + m)

def verify_with_ctx(pk, ctx: str, m: bytes, sig: int) -> bool:
    return rsa_verify(pk, ctx.encode()+b"|" + m, sig)

ctx="pay-v1|Alice->Bob|nonce=1"
sig=sign_with_ctx(sk, ctx, b"amount=10")
print(verify_with_ctx(pk, ctx, b"amount=10", sig))
print(verify_with_ctx(pk, "pay-v1|Alice->Mallory|nonce=1", b"amount=10", sig))
# 预期输出：True False



## 练习

1) 把签名对象改成 JSON 字符串（注意稳定序列化），并讨论风险。
2) 把 bits 提到 384，观察速度变化。
3) 思考：为什么真实 RSA 签名要用 PSS？（提示：抗选择消息攻击）


## 总结与延伸

- 协议安全分析离不开“对手模型”：攻击者能监听？能篡改？能冒充？
- 实现时优先使用成熟库与标准（如 TLS）；本课程实现仅用于学习。

> 下一步：学习形式化验证、协议逻辑、以及常见真实协议的攻击案例。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？

